In [ ]:
# ================================================================
# SEL 0 -- Instalasi Dependensi & Import Library
# ================================================================
!pip install -q kagglehub transformers datasets accelerate sentencepiece scikit-learn seaborn

import gc
import json as _json
import os
import random
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_curve,
    auc,
    precision_score,
    recall_score,
)

import torch
import transformers
import kagglehub
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

print("Semua library berhasil di-import.")
print(f"   PyTorch       : {torch.__version__}")
print(f"   Transformers  : {transformers.__version__}")
print(f"   GPU tersedia  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU           : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM total    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# ================================================================
# SEL 1 -- Instalasi Dependensi BERTopic (Post-Training)
# ================================================================
!pip install -q bertopic sentence-transformers umap-learn hdbscan


**Catatan BERTopic:** BERTopic dijalankan setelah training classifier IndoBERT selesai (post-training), sehingga **tidak mempengaruhi akurasi maupun kecepatan training baseline** sama sekali.

In [ ]:
# ================================================================
# SEL 2 -- Konfigurasi Utama (Config)
# ================================================================
# CONSTRAINT (tidak boleh diubah per spesifikasi):
#   train_batch_size = 96   <- optimal untuk VRAM 15 GB (fp16)
#   eval_batch_size  = 384  <- evaluasi cepat, inference-only
#   max_length       = 256  <- sesuai panjang teks dataset berita
#
# [FIX Q1] cudnn.deterministic=True & benchmark=False
#           Reproduktifitas penuh; tanpa ini cuDNN memilih kernel
#           secara non-deterministik antar-run meski seed sama.
# ================================================================
@dataclass
class Config:
    # -- Path Dataset ------------------------------------------------
    path_cnn    : str   = "/content/dataset/Summarized_CNN.csv"
    path_detik  : str   = "/content/dataset/Summarized_Detik.csv"
    path_kompas : str   = "/content/dataset/Summarized_Kompas.csv"
    path_tbh    : str   = "/content/dataset/Summarized_TurnBackHoax.csv"
    path_extra  : str   = "/content/dataset/Summarized_2020+.csv"
    # -- Model -------------------------------------------------------
    model_name       : str   = "indolem/indobert-base-uncased"
    # -- Hyperparameter Training -------------------------------------
    max_length       : int   = 256    # <- CONSTRAINT
    train_batch_size : int   = 96     # <- CONSTRAINT
    eval_batch_size  : int   = 384    # <- CONSTRAINT
    grad_accumulation: int   = 2
    learning_rate    : float = 2e-5
    weight_decay     : float = 0.01
    num_epochs       : int   = 3
    seed             : int   = 42
    # -- Balancing ---------------------------------------------------
    balance_minority : bool  = True
    # -- Output ------------------------------------------------------
    output_dir       : str   = "indobert_hoax_model_v3"

cfg = Config()
set_seed(cfg.seed)

# [FIX Q1] Reproduktifitas penuh pada operasi CUDA
torch.backends.cudnn.deterministic = True   # pilih kernel deterministik
torch.backends.cudnn.benchmark     = False  # nonaktifkan auto-tuner

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device aktif           : {device}")
print(f"cudnn.deterministic    : {torch.backends.cudnn.deterministic}   [FIX Q1]")
print(f"cudnn.benchmark        : {torch.backends.cudnn.benchmark}  [FIX Q1]")
print(f"train_batch_size       : {cfg.train_batch_size}  CONSTRAINT")
print(f"eval_batch_size        : {cfg.eval_batch_size} CONSTRAINT")
print(f"Efektif batch size     : {cfg.train_batch_size * cfg.grad_accumulation}")
print(f"max_length             : {cfg.max_length}  CONSTRAINT")


In [ ]:
# ================================================================
# SEL 3 -- Konfigurasi BERTopic (Post-Training, Opsional)
# ================================================================
# [FIX MR-3] BERTOPIC_HITUNG_PROB=False  -> fit 2-3x lebih cepat
# [FIX Q3]   NR_TOPIK="auto"             -> merge topik mirip otomatis
# ================================================================
AKTIFKAN_BERTOPIC        = True
AKTIFKAN_UNGGAH_BERTOPIC = False

DIR_OUTPUT_BERTOPIC      = "/content/bertopic_model_v1"
MODEL_EMBEDDING_BERTOPIC = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
UKURAN_BATCH_EMBED       = 32
MAKS_DOKUMEN_BERTOPIC    = None
REPO_TOPIC_HF            = "fjrmhri/TA-FINAL-BERTopic"
BERTOPIC_HITUNG_PROB     = False  # [FIX MR-3] False = jauh lebih cepat
NR_TOPIK                 = "auto" # [FIX Q3]  merge topik mirip otomatis

SEED_BERTOPIC = cfg.seed

print(f"BERTopic aktif         : {AKTIFKAN_BERTOPIC}")
print(f"Hitung probabilitas    : {BERTOPIC_HITUNG_PROB}   [FIX MR-3]")
print(f"nr_topics              : {NR_TOPIK}           [FIX Q3]")


In [ ]:
# ================================================================
# SEL 4 -- Unduh Dataset dari Kaggle
# ================================================================
kaggle_cache_dir = kagglehub.dataset_download("fjrmhri/dataset-berita")
print("Path dataset (cache kagglehub):", kaggle_cache_dir)

direktori_dataset = Path("/content/dataset")
direktori_dataset.mkdir(parents=True, exist_ok=True)

pemetaan_berkas = {
    "Summarized_CNN.csv":          ["dataset/Summarized_CNN.csv"],
    "Summarized_Detik.csv":        ["dataset/Summarized_Detik.csv"],
    "Summarized_Kompas.csv":       ["dataset/Summarized_Kompas.csv"],
    "Summarized_TurnBackHoax.csv": ["dataset/Summarized_TurnBackHoax.csv"],
    "Summarized_2020+.csv":        ["dataset/Summarized_2020+.csv"],
}

for nama_tujuan, kandidat_path in pemetaan_berkas.items():
    ditemukan = None
    for kandidat in kandidat_path:
        path_kandidat = Path(kaggle_cache_dir) / kandidat
        if path_kandidat.exists():
            ditemukan = path_kandidat
            break
    if ditemukan is None:
        raise FileNotFoundError(f"Berkas tidak ditemukan: {nama_tujuan}")
    shutil.copy(ditemukan, direktori_dataset / nama_tujuan)
    print(f"  Disalin: {ditemukan.name} -> {direktori_dataset / nama_tujuan}")

print("\nDataset siap di /content/dataset")


In [ ]:
# ================================================================
# SEL 5 -- Pemuatan Data (Data Loading)
# ================================================================
KOLOM_DASAR = [
    "url", "judul", "tanggal", "isi_berita",
    "Narasi", "Clean Narasi", "hoax", "summary",
]


def muat_satu_dataset(path: str, nama_sumber: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File tidak ditemukan: {path}")
    print(f"\nMembaca: {path}  (sumber={nama_sumber})")
    df = pd.read_csv(path)
    print("  Kolom tersedia:", list(df.columns))
    for kol in KOLOM_DASAR:
        if kol not in df.columns:
            df[kol] = "" if kol != "hoax" else np.nan
    df = df[KOLOM_DASAR].copy()
    df["source"] = nama_sumber
    return df


def muat_semua_dataset(cfg: Config) -> pd.DataFrame:
    kumpulan: List[pd.DataFrame] = []
    kumpulan.append(muat_satu_dataset(cfg.path_cnn,    "cnn"))
    kumpulan.append(muat_satu_dataset(cfg.path_detik,  "detik"))
    kumpulan.append(muat_satu_dataset(cfg.path_kompas, "kompas"))
    kumpulan.append(muat_satu_dataset(cfg.path_tbh,    "turnbackhoax"))
    kumpulan.append(muat_satu_dataset(cfg.path_extra,  "merged_extra"))
    df_gabung = pd.concat(kumpulan, ignore_index=True)
    print(f"\nTotal baris (mentah): {len(df_gabung):,}")
    return df_gabung


In [ ]:
# ================================================================
# SEL 6 -- Pra-pemrosesan & Pelabelan
# ================================================================
# [FIX HR-2] "summary" masuk rantai fallback pilih_teks setelah
#             "isi_berita" -- lebih padat semantik dari "judul".
# [FIX MR-1] Dedup ketat: teks dengan label konflik dihapus.
# ================================================================

def bangun_dataframe_training(df_mentah: pd.DataFrame) -> pd.DataFrame:
    df = df_mentah.copy()

    def pilih_teks(baris):
        # Prioritas: Clean Narasi -> Narasi -> isi_berita -> summary -> judul
        for kol in ["Clean Narasi", "Narasi", "isi_berita", "summary", "judul"]:
            nilai = baris.get(kol, "")
            if isinstance(nilai, str) and nilai.strip():
                return nilai.strip()
        return ""

    df["text"] = df.apply(pilih_teks, axis=1).astype(str).str.strip()

    n = len(df)
    df = df[df["text"] != ""].reset_index(drop=True)
    print(f"Baris tanpa teks dibuang       : {n - len(df):,}")

    df["hoax_num"] = pd.to_numeric(df["hoax"], errors="coerce")
    mask_nan = df["hoax_num"].isna()
    df.loc[(df["source"].isin(["cnn","detik","kompas"])) & mask_nan, "hoax_num"] = 0
    df.loc[(df["source"] == "turnbackhoax") & mask_nan, "hoax_num"] = 1
    df.loc[(df["source"] == "merged_extra")  & mask_nan, "hoax_num"] = 0

    n = len(df)
    df = df[df["hoax_num"].isin([0, 1])].reset_index(drop=True)
    print(f"Baris label NaN dibuang        : {n - len(df):,}")

    df["label"] = df["hoax_num"].astype(int)
    df = df.drop(columns=["hoax_num"])

    n = len(df)
    df = df.drop_duplicates(subset=["text", "label"])
    print(f"Duplikat (text+label) dibuang  : {n - len(df):,}")

    # [FIX MR-1] Hapus teks dengan label konflik
    unik_per_label = df.groupby("text")["label"].nunique()
    teks_konflik   = unik_per_label[unik_per_label > 1].index
    n = len(df)
    df = df[~df["text"].isin(teks_konflik)].reset_index(drop=True)
    print(f"Teks label-konflik dibuang     : {n - len(df):,} baris "
          f"({len(teks_konflik):,} teks unik)  [FIX MR-1]")

    print(f"\nTotal baris bersih: {len(df):,}")
    print(df["label"].value_counts().to_string())
    print(df.groupby("source")["label"].value_counts().unstack(fill_value=0).to_string())
    return df


In [ ]:
# ================================================================
# SEL 7 -- Pembagian Data & Penyeimbangan Kelas
# ================================================================
def bagi_data_stratified(df: pd.DataFrame, seed: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(
        df, test_size=0.30, stratify=df["label"], random_state=seed)
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df["label"], random_state=seed)
    print(f"  Train : {len(train_df):,} | Val : {len(val_df):,} | Test : {len(test_df):,}")
    return (train_df.reset_index(drop=True),
            val_df.reset_index(drop=True),
            test_df.reset_index(drop=True))


def seimbangkan_kelas_minoritas(train_df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    jumlah = train_df["label"].value_counts()
    print("Distribusi TRAIN sebelum balancing:"); print(jumlah.to_string())
    if len(jumlah) != 2:
        print("Label tidak biner, balancing dilewati."); return train_df
    n_maks = jumlah.max()
    kumpulan = []
    for lbl, df_lbl in train_df.groupby("label"):
        if len(df_lbl) < n_maks:
            df_lbl = resample(df_lbl, replace=True, n_samples=n_maks, random_state=seed)
        kumpulan.append(df_lbl)
    hasil = pd.concat(kumpulan, ignore_index=True)
    hasil = hasil.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print("Distribusi TRAIN setelah balancing:"); print(hasil["label"].value_counts().to_string())
    return hasil


## Visualisasi Distribusi Dataset [A1]

Grafik distribusi label, sampel per sumber, panjang teks, dan label per sumber. Dipanggil setelah `df_bersih` tersedia di SEL 12.

In [ ]:
# ================================================================
# SEL 8 -- Visualisasi Distribusi Dataset  [A1 -- Nice-to-have]
# ================================================================
def visualisasi_distribusi_dataset(df: pd.DataFrame, judul_prefix: str = ""):
    df = df.copy()
    df["panjang_teks"] = df["text"].astype(str).str.len()
    nama_label = {0: "Non-Hoax", 1: "Hoax"}

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"Distribusi Dataset Deteksi Hoaks{' -- ' + judul_prefix if judul_prefix else ''}",
        fontsize=14, fontweight="bold", y=1.01)

    # Plot 1: Distribusi Label Global
    hitung_label = df["label"].value_counts().sort_index()
    warna_label  = ["#16A34A", "#DC2626"]
    batang = axes[0,0].bar(
        [nama_label[i] for i in hitung_label.index],
        hitung_label.values, color=warna_label, edgecolor="white")
    for b, v in zip(batang, hitung_label.values):
        axes[0,0].text(b.get_x()+b.get_width()/2, b.get_height()+20,
                        f"{v:,}", ha="center", va="bottom", fontsize=11, fontweight="bold")
    axes[0,0].set_title("Distribusi Label Global", fontsize=11, fontweight="bold")
    axes[0,0].set_ylabel("Jumlah Sampel")
    axes[0,0].set_ylim(0, hitung_label.max()*1.15)
    axes[0,0].grid(axis="y", alpha=0.3)

    # Plot 2: Jumlah Sampel per Sumber
    hitung_sumber = df["source"].value_counts()
    warna_sumber  = plt.cm.Set2(np.linspace(0, 1, len(hitung_sumber)))
    batang2 = axes[0,1].bar(hitung_sumber.index, hitung_sumber.values,
                              color=warna_sumber, edgecolor="white")
    for b, v in zip(batang2, hitung_sumber.values):
        axes[0,1].text(b.get_x()+b.get_width()/2, b.get_height()+10,
                        f"{v:,}", ha="center", va="bottom", fontsize=9)
    axes[0,1].set_title("Jumlah Sampel per Sumber", fontsize=11, fontweight="bold")
    axes[0,1].set_ylabel("Jumlah Sampel")
    axes[0,1].tick_params(axis="x", rotation=20)
    axes[0,1].grid(axis="y", alpha=0.3)

    # Plot 3: Distribusi Panjang Teks
    for lbl, warna in [(0,"#16A34A"),(1,"#DC2626")]:
        subset = df[df["label"]==lbl]["panjang_teks"]
        axes[1,0].hist(subset, bins=50, alpha=0.65, color=warna,
                        label=nama_label[lbl], edgecolor="none")
    axes[1,0].axvline(df["panjang_teks"].median(), color="#1D4ED8",
                       ls="--", lw=1.5,
                       label=f"Median = {df['panjang_teks'].median():.0f}")
    axes[1,0].set_title("Distribusi Panjang Teks (karakter)", fontsize=11, fontweight="bold")
    axes[1,0].set_xlabel("Panjang Teks"); axes[1,0].set_ylabel("Frekuensi")
    axes[1,0].legend(fontsize=9); axes[1,0].grid(axis="y", alpha=0.3)

    # Plot 4: Label per Sumber (Stacked Bar)
    tabel = df.groupby(["source","label"]).size().unstack(fill_value=0)
    tabel.columns = [nama_label[c] for c in tabel.columns]
    tabel.plot(kind="bar", ax=axes[1,1], color=["#16A34A","#DC2626"],
               edgecolor="white", linewidth=0.8)
    axes[1,1].set_title("Distribusi Label per Sumber", fontsize=11, fontweight="bold")
    axes[1,1].set_xlabel(""); axes[1,1].set_ylabel("Jumlah Sampel")
    axes[1,1].tick_params(axis="x", rotation=20)
    axes[1,1].legend(title="Label"); axes[1,1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("distribusi_dataset.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Grafik disimpan ke: distribusi_dataset.png")
    print(f"\nStatistik Panjang Teks:")
    print(df["panjang_teks"].describe().round(1).to_string())
    print(f"\nRasio Hoax: {df['label'].mean()*100:.1f}%")


In [ ]:
# ================================================================
# SEL 9 -- Login Hugging Face
# ================================================================
from huggingface_hub import login
login()


In [ ]:
# ================================================================
# SEL 10 -- Tokenisasi & Konversi ke HuggingFace Dataset
# ================================================================
# [FIX LR-5] Bug __index_level removal diperbaiki dengan fungsi
#             helper hapus_kolom_index() yang mengembalikan dataset baru.
# ================================================================
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)


def tokenisasi_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)


def hapus_kolom_index(ds: Dataset) -> Dataset:
    cols_idx = [c for c in ds.column_names if c.startswith("__index_level")]
    return ds.remove_columns(cols_idx) if cols_idx else ds


def siapkan_dataset_hf(
    train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame
) -> Tuple[Dataset, Dataset, Dataset]:
    kol = ["text", "label", "source"]
    ds_train = hapus_kolom_index(Dataset.from_pandas(train_df[kol]))
    ds_val   = hapus_kolom_index(Dataset.from_pandas(val_df[kol]))
    ds_test  = hapus_kolom_index(Dataset.from_pandas(test_df[kol]))

    ds_train = ds_train.map(tokenisasi_batch, batched=True)
    ds_val   = ds_val.map(tokenisasi_batch,   batched=True)
    ds_test  = ds_test.map(tokenisasi_batch,  batched=True)

    kol_m = ["input_ids","attention_mask","label"]
    ds_train.set_format(type="torch", columns=kol_m)
    ds_val.set_format(type="torch",   columns=kol_m)
    ds_test.set_format(type="torch",  columns=kol_m)

    print(f"Train: {len(ds_train):,} | Val: {len(ds_val):,} | Test: {len(ds_test):,}")
    return ds_train, ds_val, ds_test


In [ ]:
# ================================================================
# SEL 11 -- Definisi Model, Metrik & Konfigurasi Training
# ================================================================
# [FIX LR-1] evaluation_strategy="epoch", save_strategy="epoch",
#             load_best_model_at_end=True, metric_for_best_model="f1"
# [FIX LR-3] warmup_ratio=0.06, lr_scheduler_type="linear"
# [FIX LR-1] report_to="none" -- nonaktifkan wandb/mlflow
# [FIX Q2]   dataloader_num_workers=0 -- cegah deadlock di Colab
# [FIX O1]   group_by_length=True -- +10-15% kecepatan training
# [FIX O3]   dataloader_pin_memory=True -- transfer CPU->GPU lebih cepat
# ================================================================
label_ke_id = {"not_hoax": 0, "hoax": 1}
id_ke_label = {v: k for k, v in label_ke_id.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=2,
    id2label=id_ke_label,
    label2id={v: k for k, v in id_ke_label.items()},
).to(device)

collator_padding = DataCollatorWithPadding(tokenizer=tokenizer)


def hitung_metrik(eval_pred):
    logits, labels = eval_pred
    prediksi = np.argmax(logits, axis=-1)
    akurasi  = accuracy_score(labels, prediksi)
    prec_b, rec_b, f1_b, _ = precision_recall_fscore_support(
        labels, prediksi, average="binary", pos_label=1)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        labels, prediksi, average="weighted")
    return {
        "accuracy":           float(akurasi),
        "precision":          float(prec_b),
        "recall":             float(rec_b),
        "f1":                 float(f1_b),
        "precision_weighted": float(prec_w),
        "recall_weighted":    float(rec_w),
        "f1_weighted":        float(f1_w),
    }


argumen_training = TrainingArguments(
    output_dir                  = cfg.output_dir,
    per_device_train_batch_size = cfg.train_batch_size,   # 96 CONSTRAINT
    per_device_eval_batch_size  = cfg.eval_batch_size,    # 384 CONSTRAINT
    gradient_accumulation_steps = cfg.grad_accumulation,
    num_train_epochs            = cfg.num_epochs,
    learning_rate               = cfg.learning_rate,
    weight_decay                = cfg.weight_decay,
    warmup_ratio                = 0.06,        # [FIX LR-3]
    lr_scheduler_type           = "linear",    # [FIX LR-3]
    evaluation_strategy         = "epoch",     # [FIX LR-1]
    save_strategy               = "epoch",     # [FIX LR-1]
    load_best_model_at_end      = True,        # [FIX LR-1]
    metric_for_best_model       = "f1",        # [FIX LR-1]
    greater_is_better           = True,
    logging_steps               = 50,
    save_total_limit            = 2,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 0,           # [FIX Q2] aman di Colab
    dataloader_pin_memory       = True,        # [FIX O3]
    group_by_length             = True,        # [FIX O1] +10-15% speed
    do_train                    = True,
    do_eval                     = True,
    seed                        = cfg.seed,
    report_to                   = "none",      # [FIX LR-1]
)

trainer = Trainer(
    model=model, args=argumen_training,
    train_dataset=None, eval_dataset=None,
    data_collator=collator_padding, compute_metrics=hitung_metrik,
)

print("Model & Trainer siap.")
print(f"   Best model criterion  : {argumen_training.metric_for_best_model}")
print(f"   group_by_length       : {argumen_training.group_by_length}  [FIX O1]")
print(f"   dataloader_num_workers: {argumen_training.dataloader_num_workers}   [FIX Q2]")
print(f"   dataloader_pin_memory : {argumen_training.dataloader_pin_memory}  [FIX O3]")


In [ ]:
# ================================================================
# SEL 12 -- Pipeline End-to-End: Muat -> Proses -> Latih -> Evaluasi
# ================================================================
# [FIX MR-2] train_df_pra_oversampling disimpan sebelum balancing
# [FIX B2]   _cache_prediksi.clear() dipanggil setelah training
# ================================================================

df_mentah = muat_semua_dataset(cfg)
df_bersih = bangun_dataframe_training(df_mentah)

# [A1] Visualisasi distribusi dataset
visualisasi_distribusi_dataset(df_bersih, judul_prefix="Setelah Pembersihan")

train_df, val_df, test_df = bagi_data_stratified(df_bersih, seed=cfg.seed)

# [FIX MR-2] Simpan corpus pra-oversampling untuk BERTopic
train_df_pra_oversampling = train_df.copy()
print(f"\n[FIX MR-2] Corpus BERTopic (pra-oversampling): {len(train_df_pra_oversampling):,}")

if cfg.balance_minority:
    train_df = seimbangkan_kelas_minoritas(train_df, seed=cfg.seed)

ds_train, ds_val, ds_test = siapkan_dataset_hf(train_df, val_df, test_df)

trainer.train_dataset = ds_train
trainer.eval_dataset  = ds_val

print("\n" + "="*60)
print("Memulai training IndoBERT...")
print("="*60)
hasil_training = trainer.train()

print(f"\nTraining selesai.")
print(f"   Checkpoint terbaik : {trainer.state.best_model_checkpoint}")
print(f"   Metrik terbaik (f1): {trainer.state.best_metric:.4f}")

log_history_training = list(trainer.state.log_history)

# [FIX B2] Reset cache prediksi agar tidak stale jika sel ini dijalankan ulang
_cache_prediksi: Dict[str, object] = {}
print("\n[FIX B2] Cache prediksi direset setelah training baru selesai.")

print("\n" + "="*60 + "\nEvaluasi VALIDATION SET\n" + "="*60)
metrik_val = trainer.evaluate(ds_val); print(metrik_val)

print("\n" + "="*60 + "\nEvaluasi TEST SET\n" + "="*60)
metrik_test = trainer.evaluate(ds_test); print(metrik_test)


In [ ]:
# ================================================================
# SEL 13 -- Visualisasi Loss Curve & Metrik per Epoch
# ================================================================
def plot_kurva_loss(log_history: list):
    l_step, l_loss = [], []
    e_step, e_loss, e_epoch, e_f1, e_acc = [], [], [], [], []
    for entri in log_history:
        if "loss" in entri and "eval_loss" not in entri:
            l_step.append(entri["step"]); l_loss.append(entri["loss"])
        if "eval_loss" in entri:
            e_step.append(entri["step"]); e_loss.append(entri["eval_loss"])
            if "epoch"        in entri: e_epoch.append(entri["epoch"])
            if "eval_f1"      in entri: e_f1.append(entri["eval_f1"])
            if "eval_accuracy"in entri: e_acc.append(entri["eval_accuracy"])

    if not l_step and not e_step:
        print("Log history kosong."); return

    n_plot = 1 + int(bool(e_step)) + int(bool(e_f1))
    fig, axes = plt.subplots(1, n_plot, figsize=(6*n_plot, 4))
    if n_plot == 1: axes = [axes]

    if l_step:
        axes[0].plot(l_step, l_loss, color="#2563EB", lw=1.5, label="Train Loss")
        axes[0].set_title("Training Loss per Step", fontweight="bold"); axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    if e_step:
        x, xl = (e_epoch,"Epoch") if e_epoch else (e_step,"Step")
        axes[1].plot(x, e_loss, "o-", color="#DC2626", lw=2, ms=6, label="Eval Loss")
        axes[1].set_title("Eval Loss per Epoch", fontweight="bold"); axes[1].set_xlabel(xl)
        axes[1].set_ylabel("Loss"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    if e_f1:
        x, xl = (e_epoch,"Epoch") if e_epoch else (list(range(1,len(e_f1)+1)),"Checkpoint")
        axes[2].plot(x, e_f1, "s-", color="#16A34A", lw=2, ms=6, label="F1 (hoax)")
        if e_acc:
            axes[2].plot(x, e_acc, "^--", color="#D97706", lw=1.5, ms=5, label="Accuracy")
        axes[2].set_title("Metrik per Epoch", fontweight="bold"); axes[2].set_xlabel(xl)
        axes[2].set_ylabel("Score"); axes[2].set_ylim(0, 1.05)
        axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle("Ringkasan Training IndoBERT -- Hoax Detection", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("kurva_training.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Grafik disimpan ke: kurva_training.png")


plot_kurva_loss(log_history_training)


In [ ]:
# ================================================================
# SEL 14 -- Laporan Detail & Confusion Matrix
# ================================================================

def dapatkan_prediksi(trainer: Trainer, dataset: Dataset, nama_split: str):
    if nama_split not in _cache_prediksi:
        _cache_prediksi[nama_split] = trainer.predict(dataset)
    return _cache_prediksi[nama_split]


def laporan_detail(trainer: Trainer, dataset: Dataset, nama_split: str):
    out    = dapatkan_prediksi(trainer, dataset, nama_split)
    y_asli = out.label_ids
    y_pred = np.argmax(out.predictions, axis=-1)
    kelas  = ["not_hoax", "hoax"]

    print(f"\n{'='*60}\n  Laporan Klasifikasi -- {nama_split}\n{'='*60}")
    print(classification_report(y_asli, y_pred, target_names=kelas, digits=4))

    cm      = confusion_matrix(y_asli, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=kelas, yticklabels=kelas, ax=axes[0],
                linewidths=0.5, annot_kws={"size":13,"weight":"bold"})
    axes[0].set_title(f"Confusion Matrix (Jumlah)\n{nama_split}", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Prediksi"); axes[0].set_ylabel("Aktual")

    sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Greens",
                xticklabels=kelas, yticklabels=kelas, ax=axes[1],
                linewidths=0.5, vmin=0, vmax=1, annot_kws={"size":13})
    axes[1].set_title(f"Confusion Matrix (Proporsi Recall)\n{nama_split}", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Prediksi"); axes[1].set_ylabel("Aktual")

    plt.tight_layout()
    fname = f"confusion_matrix_{nama_split.lower().replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
    print(f"Confusion matrix disimpan ke: {fname}")
    return y_asli, y_pred, out.predictions


laporan_detail(trainer, ds_val,  "Validation")
laporan_detail(trainer, ds_test, "Test")


In [ ]:
# ================================================================
# SEL 15 -- Kurva ROC & AUC
# ================================================================
def plot_kurva_roc(trainer, dataset, nama_split, warna="#2563EB"):
    out   = dapatkan_prediksi(trainer, dataset, nama_split)
    probs = torch.softmax(torch.tensor(out.predictions, dtype=torch.float32), dim=-1).numpy()
    fpr, tpr, _ = roc_curve(out.label_ids, probs[:,1], pos_label=1)
    nilai_auc = auc(fpr, tpr)

    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, color=warna, lw=2, label=f"AUC = {nilai_auc:.4f}")
    plt.plot([0,1],[0,1],"k--", lw=1, label="Random Baseline")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"Kurva ROC -- {nama_split}", fontsize=12, fontweight="bold")
    plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.tight_layout()
    fname = f"roc_curve_{nama_split.lower().replace(' ','_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight"); plt.show()
    print(f"ROC curve disimpan ke: {fname}  |  AUC = {nilai_auc:.4f}")
    return nilai_auc


auc_val  = plot_kurva_roc(trainer, ds_val,  "Validation", warna="#2563EB")
auc_test = plot_kurva_roc(trainer, ds_test, "Test",       warna="#DC2626")


In [ ]:
# ================================================================
# SEL 16 -- Kalibrasi Threshold Optimal  [FIX HR-1]
# ================================================================
def kalibrasi_threshold(trainer, dataset, nama_split="val") -> float:
    out    = dapatkan_prediksi(trainer, dataset, nama_split)
    y_asli = out.label_ids
    probs  = torch.softmax(torch.tensor(out.predictions, dtype=torch.float32), dim=-1).numpy()
    p_hoax = probs[:,1]

    sweep    = np.arange(0.30, 0.80, 0.01)
    riwayat  = [(float(t), float(f1_score(y_asli,(p_hoax>=t).astype(int),
                                           pos_label=1, zero_division=0)))
                for t in sweep]
    t_best, f1_best = max(riwayat, key=lambda x: x[1])

    tv, fv = zip(*riwayat)
    plt.figure(figsize=(7, 4))
    plt.plot(tv, fv, color="#2563EB", lw=2)
    plt.axvline(t_best, color="#DC2626", ls="--", lw=1.5,
                label=f"Optimal = {t_best:.2f}  (F1={f1_best:.4f})")
    plt.axvline(0.5, color="#6B7280", ls=":", lw=1.2, label="Default = 0.50")
    plt.xlabel("Threshold p(hoax)"); plt.ylabel("F1-score (hoax)")
    plt.title(f"Kalibrasi Threshold -- {nama_split}", fontsize=12, fontweight="bold")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(f"kalibrasi_threshold_{nama_split}.png", dpi=150, bbox_inches="tight")
    plt.show()

    f1_def = f1_score(y_asli, (p_hoax>=0.5).astype(int), pos_label=1)
    print(f"[{nama_split}] Threshold optimal : {t_best:.2f}")
    print(f"[{nama_split}] F1 hoax @optimal  : {f1_best:.4f}")
    print(f"[{nama_split}] F1 hoax @0.50     : {f1_def:.4f}")
    return t_best


THRESHOLD_OPTIMAL = kalibrasi_threshold(trainer, ds_val, "validation")

out_test = dapatkan_prediksi(trainer, ds_test, "Test")
p_test   = torch.softmax(torch.tensor(out_test.predictions, dtype=torch.float32), dim=-1).numpy()[:,1]
y_test   = out_test.label_ids

f1_def = f1_score(y_test, (p_test>=0.5).astype(int), pos_label=1)
f1_opt = f1_score(y_test, (p_test>=THRESHOLD_OPTIMAL).astype(int), pos_label=1)
print(f"\nTest F1 @0.50     : {f1_def:.4f}")
print(f"Test F1 @{THRESHOLD_OPTIMAL:.2f} (opt): {f1_opt:.4f}")
print(f"\nTHRESHOLD_OPTIMAL = {THRESHOLD_OPTIMAL:.2f}")


## Tabel Perbandingan Multi-Threshold [A2]

Tabel Precision/Recall/F1 pada berbagai threshold untuk memilih trade-off false positive vs. false negative.

In [ ]:
# ================================================================
# SEL 17 -- Tabel Perbandingan Multi-Threshold  [A2 -- Nice-to-have]
# ================================================================
def tabel_perbandingan_threshold(trainer, dataset, nama_split, threshold_optimal,
                                  threshold_list=None):
    if threshold_list is None:
        threshold_list = [0.30, 0.35, 0.40, 0.45, 0.50,
                          0.55, 0.60, 0.65, 0.70, threshold_optimal]
    threshold_list = sorted(set(round(t, 2) for t in threshold_list))

    out   = dapatkan_prediksi(trainer, dataset, nama_split)
    y     = out.label_ids
    probs = torch.softmax(torch.tensor(out.predictions, dtype=torch.float32), dim=-1).numpy()
    p_h   = probs[:,1]

    baris = []
    for t in threshold_list:
        pred = (p_h >= t).astype(int)
        prec = precision_score(y, pred, pos_label=1, zero_division=0)
        rec  = recall_score(y, pred,    pos_label=1, zero_division=0)
        f1   = f1_score(y, pred,        pos_label=1, zero_division=0)
        tp   = int(((pred==1)&(y==1)).sum())
        fp   = int(((pred==1)&(y==0)).sum())
        fn   = int(((pred==0)&(y==1)).sum())
        ket  = ""
        if abs(t - 0.50) < 0.005:            ket = "<- default"
        if abs(t - threshold_optimal) < 0.005: ket = "<- OPTIMAL"
        baris.append({"Threshold": f"{t:.2f}", "Precision": f"{prec:.4f}",
                       "Recall": f"{rec:.4f}", "F1-Hoax": f"{f1:.4f}",
                       "TP": tp, "FP": fp, "FN": fn, "Keterangan": ket})

    df_t = pd.DataFrame(baris)
    print(f"\n{'='*70}\n  Tabel Perbandingan Threshold -- {nama_split}\n{'='*70}")
    print(df_t.to_string(index=False))
    print("="*70)
    return df_t


tabel_val  = tabel_perbandingan_threshold(trainer, ds_val,  "Validation", THRESHOLD_OPTIMAL)
tabel_test = tabel_perbandingan_threshold(trainer, ds_test, "Test",       THRESHOLD_OPTIMAL)


In [ ]:
# ================================================================
# SEL 18 -- Simpan Model Terbaik, Tokenizer & Metadata Inferensi
# ================================================================
# [FIX Q4] Metadata menyertakan versi library & tanggal training.
# ================================================================
os.makedirs(cfg.output_dir, exist_ok=True)
trainer.model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

metadata_inferensi = {
    "threshold_optimal":     float(THRESHOLD_OPTIMAL),
    "threshold_default":     0.5,
    "metric_for_best_model": "f1",
    "model_name":            cfg.model_name,
    "max_length":            cfg.max_length,
    "label2id":              label_ke_id,
    "id2label":              {str(k): v for k, v in id_ke_label.items()},
    "transformers_version":  transformers.__version__,   # [FIX Q4]
    "torch_version":         torch.__version__,          # [FIX Q4]
    "training_date":         pd.Timestamp.now().strftime("%Y-%m-%d"),  # [FIX Q4]
}
with open(os.path.join(cfg.output_dir, "inference_config.json"), "w",
          encoding="utf-8") as _f:
    _json.dump(metadata_inferensi, _f, indent=2, ensure_ascii=False)

print(f"Model & metadata disimpan ke: {cfg.output_dir}/")
print(f"   Best checkpoint      : {trainer.state.best_model_checkpoint}")
print(f"   THRESHOLD_OPTIMAL    : {THRESHOLD_OPTIMAL:.2f}")
print(f"   transformers_version : {transformers.__version__}  [FIX Q4]")
print(f"   training_date        : {metadata_inferensi['training_date']}  [FIX Q4]")


In [ ]:
# ================================================================
# SEL 19 -- Unduh Model ke Lokal (Opsional)
# ================================================================
from google.colab import files
nama_zip = f"{cfg.output_dir}.zip"
shutil.make_archive(cfg.output_dir, "zip", cfg.output_dir)
files.download(nama_zip)
print(f"Unduhan dimulai: {nama_zip}")


## Inferensi Multi-Paragraf (Sentence-Level + Topic)

**Tidak mengubah training classifier IndoBERT.** Semua proses adalah post-processing:
memisahkan teks menjadi paragraf dan kalimat, mengklasifikasikan setiap kalimat, mengagregasi label, dan mengekstraksi topik TF-IDF.

- **[FIX HR-1]** Menggunakan `THRESHOLD_OPTIMAL` (dari val set)
- **[FIX O2]** `torch.inference_mode()` menggantikan `torch.no_grad()`

In [ ]:
# ================================================================
# SEL 20 -- Utilitas Inferensi Multi-Paragraf (Post-Processing)
# ================================================================
# [FIX HR-1] prediksi_batch() pakai THRESHOLD_OPTIMAL (fallback 0.5)
# [FIX O2]   torch.inference_mode() -- lebih efisien dari no_grad
# ================================================================
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

REGEX_PISAH_PARAGRAF = re.compile(r"(?:?
){2,}")
REGEX_PISAH_KALIMAT  = re.compile(r'[^.!?]+(?:[.!?]+(?:[")\]]+)?)|[^.!?]+$')
REGEX_SPASI          = re.compile(r"\s+")
REGEX_TOKEN          = re.compile(r"[a-zA-Z]{3,}")

STOPWORDS_ID = {
    "yang","dan","atau","di","ke","dari","untuk","dengan","pada",
    "adalah","itu","ini","dalam","sebagai","karena","juga","agar",
    "oleh","saat","akan","telah","sudah","tidak","iya","ya","kita",
    "mereka","kami","anda","hingga","lebih","masih","dapat","bisa",
    "setelah","sebelum","tersebut","terhadap","disebut","menurut",
}


def normalisasi_teks(teks: str) -> str:
    return REGEX_SPASI.sub(" ", str(teks)).strip()


def pisahkan_paragraf(teks: str) -> List[str]:
    mentah = str(teks).strip()
    if not mentah: return []
    par = [p.strip() for p in REGEX_PISAH_PARAGRAF.split(mentah) if p.strip()]
    return par if par else [mentah]


def pisahkan_kalimat(paragraf: str) -> List[str]:
    d = normalisasi_teks(paragraf)
    if not d: return []
    kal = [m.group(0).strip() for m in REGEX_PISAH_KALIMAT.finditer(d)]
    return [k for k in kal if k] or [d]


def prediksi_batch(teks_list: List[str], ukuran_batch: int = 64) -> List[Dict]:
    if not teks_list: return []
    th = THRESHOLD_OPTIMAL if "THRESHOLD_OPTIMAL" in globals() else 0.5
    semua_probs = []
    model.eval()
    for i in range(0, len(teks_list), ukuran_batch):
        potongan = [normalisasi_teks(t) or "[EMPTY]" for t in teks_list[i:i+ukuran_batch]]
        enc = tokenizer(potongan, truncation=True, max_length=cfg.max_length,
                        padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.inference_mode():   # [FIX O2]
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
        semua_probs.extend(probs)

    hasil = []
    for baris in semua_probs:
        p_n, p_h = float(baris[0]), float(baris[1])
        label = "hoax" if p_h >= th else "not_hoax"
        conf  = max(p_h, p_n)
        warna = "red" if label=="hoax" else ("amber" if conf<0.70 else "green")
        hasil.append({
            "label": label,
            "probabilities": {"not_hoax": round(p_n,6), "hoax": round(p_h,6)},
            "hoax_probability": round(p_h, 6),
            "confidence":       round(conf, 6),
            "color":            warna,
            "threshold_used":   round(th, 2),
        })
    return hasil


def ekstrak_topik_tfidf(teks_par: List[str], top_k: int = 3, maks_fitur: int = 1500) -> List[Dict]:
    if not teks_par: return []
    vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), max_features=maks_fitur,
                           token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",
                           stop_words=list(STOPWORDS_ID))
    out_t = []
    try:
        M = vec.fit_transform(teks_par); F = vec.get_feature_names_out()
        for i in range(M.shape[0]):
            br = M.getrow(i)
            if br.nnz == 0:
                tok = [t for t in REGEX_TOKEN.findall(teks_par[i].lower()) if t not in STOPWORDS_ID]
                kk  = list(dict.fromkeys(tok))[:top_k] or ["topik_umum"]
                skor = 0.0
            else:
                pas  = sorted(zip(br.indices, br.data), key=lambda x:(-x[1],F[x[0]]))[:top_k]
                kk   = [F[idx] for idx,_ in pas]
                skor = float(np.mean([v for _,v in pas])) if pas else 0.0
            out_t.append({"label": " / ".join(kk[:2]) if kk else "topik_umum",
                           "score": round(skor,6), "keywords": kk[:top_k]})
    except Exception:
        for t in teks_par:
            tok = [x for x in REGEX_TOKEN.findall(t.lower()) if x not in STOPWORDS_ID]
            kk  = list(dict.fromkeys(tok))[:top_k] or ["topik_umum"]
            out_t.append({"label": " / ".join(kk[:2]) if kk else "topik_umum",
                           "score": 0.0, "keywords": kk[:top_k]})
    return out_t


def analisis_multi_paragraf(teks: str, ukuran_batch_kalimat: int = 64) -> Dict[str,Any]:
    teks_m = str(teks).strip()
    th = THRESHOLD_OPTIMAL if "THRESHOLD_OPTIMAL" in globals() else 0.5
    if not teks_m:
        return {"document":{"label":"not_hoax","hoax_probability":0.0,"confidence":0.0,
                             "risk_level":"low","sentence_aggregate_label":"not_hoax",
                             "threshold_used":th,
                             "summary":{"paragraph_count":0,"sentence_count":0,
                                        "hoax_sentence_count":0,"not_hoax_sentence_count":0}},
                "paragraphs":[],"shared_topics":[]}

    pred_doc = prediksi_batch([teks_m], ukuran_batch=1)[0]
    par_list = pisahkan_paragraf(teks_m)
    t_kal, p_kal = [], []
    for pi, par in enumerate(par_list):
        for si, k in enumerate(pisahkan_kalimat(par)):
            t_kal.append(k); p_kal.append((pi,si))

    pred_kal = prediksi_batch(t_kal, ukuran_batch=ukuran_batch_kalimat)
    obj_par  = [{"paragraph_index":i,"text":p,"sentences":[]} for i,p in enumerate(par_list)]
    for (pi,si),tk,pred in zip(p_kal,t_kal,pred_kal):
        obj_par[pi]["sentences"].append({"sentence_index":si,"text":tk,**pred})

    topik_list = ekstrak_topik_tfidf([p["text"] for p in obj_par])
    jh = jnh = 0
    for i,op in enumerate(obj_par):
        kal = op["sentences"]
        if not kal:
            op.update({"label":"not_hoax","hoax_probability":0.0,"confidence":0.0})
        else:
            op["label"]            = "hoax" if any(k["label"]=="hoax" for k in kal) else "not_hoax"
            op["hoax_probability"] = round(max(k["hoax_probability"] for k in kal),6)
            op["confidence"]       = round(max(k["confidence"]       for k in kal),6)
            jh  += sum(1 for k in kal if k["label"]=="hoax")
            jnh += sum(1 for k in kal if k["label"]!="hoax")
        op["topic"] = topik_list[i]

    tmap = defaultdict(list)
    for p in obj_par: tmap[p["topic"]["label"]].append(p["paragraph_index"])
    topik_bersama = sorted([{"label":l,"paragraph_indices":v} for l,v in tmap.items() if len(v)>1],
                            key=lambda x:(x["paragraph_indices"][0],x["label"]))

    return {
        "document": {
            "label":                    pred_doc["label"],
            "hoax_probability":         pred_doc["hoax_probability"],
            "confidence":               pred_doc["confidence"],
            "threshold_used":           pred_doc["threshold_used"],
            "risk_level": ("high" if pred_doc["hoax_probability"]>0.98 else
                           "medium" if pred_doc["hoax_probability"]>0.60 else "low"),
            "sentence_aggregate_label": "hoax" if jh>0 else "not_hoax",
            "summary":{"paragraph_count":len(obj_par),"sentence_count":len(pred_kal),
                        "hoax_sentence_count":jh,"not_hoax_sentence_count":jnh},
        },
        "paragraphs":obj_par, "shared_topics":topik_bersama,
    }


print("Utilitas inferensi multi-paragraf siap.")
print(f"   Threshold aktif  : {THRESHOLD_OPTIMAL:.2f}  [FIX HR-1]")
print(f"   Inference context: torch.inference_mode()  [FIX O2]")


## Demo Inferensi V1

Deteksi multi-paragraf dengan threshold yang sudah dikalibrasi.

In [ ]:
# ================================================================
# SEL 21 -- Demo Inferensi Multi-Paragraf
# ================================================================
contoh_teks = """
Beredar unggahan media sosial yang menyebut pemerintah membagikan bantuan tunai langsung tanpa syarat lewat tautan tertentu. Banyak akun meminta warga mengisi data pribadi dan OTP agar dana cair hari itu juga.

Kementerian terkait kemudian merilis klarifikasi bahwa informasi tersebut tidak benar dan meminta masyarakat tidak membagikan tautan yang tidak berasal dari kanal resmi. Warga diminta cek pengumuman hanya lewat situs resmi pemerintah.

Sejumlah komentar pengguna juga menuliskan pengalaman bahwa tautan serupa berujung pada permintaan data rekening. Pakar keamanan digital menyarankan verifikasi sumber sebelum menyebarkan pesan berantai.
"""

hasil = analisis_multi_paragraf(contoh_teks, ukuran_batch_kalimat=64)

print("=" * 60 + "\nHASIL DOKUMEN:")
for k, v in hasil["document"].items():
    if k != "summary": print(f"  {k:30s}: {v}")
print(f"  {'summary':30s}: {hasil['document']['summary']}")
print("=" * 60)

for par in hasil["paragraphs"]:
    print(f"\n[Paragraf {par['paragraph_index']}]  label={par['label']}")
    print(f"  Topik : {par['topic']['label']} (score={par['topic']['score']})")
    for k in par["sentences"]:
        print(f"  . ({k['sentence_index']}) {k['label']} | p_hoax={k['hoax_probability']:.4f} | conf={k['confidence']:.4f}")

if hasil["shared_topics"]:
    print("\nTopik Bersama:", hasil["shared_topics"])


## No-Regression Guarantee

1. **Training core tidak diubah secara semantik** -- alur `70/15/15 split -> balancing hanya di train -> classifier biner IndoBERT` identik dengan baseline.
2. **BERTopic murni post-processing** -- tidak menyentuh bobot model classifier.
3. **Semua perbaikan bersifat additive** -- bug fix + optimasi tanpa mengganti arsitektur.
4. **Threshold dikalibrasi dari val set** -- test set tidak disentuh sampai evaluasi akhir.

In [ ]:
# ================================================================
# SEL 22 -- Unggah Model ke Hugging Face (TA-FINAL)
# ================================================================
!pip install -q huggingface_hub

from huggingface_hub import login, HfApi
from google.colab import userdata

try:
    login(token=userdata.get("HF_TOKEN"))
except Exception:
    print("Token HF_TOKEN tidak ditemukan pada Colab Secrets.")

api_hf = HfApi()
id_repo = "fjrmhri/TA-FINAL"
api_hf.create_repo(repo_id=id_repo, private=False, exist_ok=True)
print(f"Mengunggah ke {id_repo} ...")
api_hf.upload_folder(
    folder_path    = f"/content/{cfg.output_dir}",
    repo_id        = id_repo,
    repo_type      = "model",
    commit_message = "Upload IndoBERT hoax detection (V1-Fix: best-model, threshold-calibrated)",
)
print(f"Selesai: https://huggingface.co/{id_repo}")


## BERTopic Topic Modeling (Post-Training)

Berjalan **setelah** training classifier selesai. Tidak masuk ke loop training.

**Perbaikan aktif:**
- **[FIX MR-2]** Corpus = `train_df_pra_oversampling` (bebas duplikat)
- **[FIX MR-3]** `calculate_probabilities=False` -- fit 2-3x lebih cepat
- **[FIX LR-4]** `sentence_embedder` dihapus dari GPU setelah encoding
- **[FIX Q3]** `nr_topics="auto"` -- merge micro-topik otomatis
- **[FIX B1]** Regex pisah paragraf diperbaiki: `r"(?:\r?\n){2,}"`
- **[A3]** Analisis topik per label hoax/non-hoax ditambahkan

In [ ]:
# ================================================================
# SEL 23 -- Import Stack BERTopic
# ================================================================
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import re as _re
print("Import BERTopic stack selesai.")


In [ ]:
# ================================================================
# SEL 24 -- Fit BERTopic pada Korpus Training (Post-Training)
# ================================================================
# [FIX MR-2] Corpus pra-oversampling (bebas duplikat)
# [FIX MR-3] calculate_probabilities=False
# [FIX LR-4] Hapus embedder dari GPU setelah encoding
# [FIX Q3]   nr_topics="auto"
# ================================================================

if not AKTIFKAN_BERTOPIC:
    model_topik = None; dokumen_topik = []; id_topik = []; prob_topik = None
    print("AKTIFKAN_BERTOPIC=False, proses BERTopic dilewati.")
else:
    # [FIX MR-2]
    dokumen_topik = train_df_pra_oversampling["text"].astype(str).tolist()
    if MAKS_DOKUMEN_BERTOPIC and len(dokumen_topik) > MAKS_DOKUMEN_BERTOPIC:
        rng = random.Random(SEED_BERTOPIC)
        dokumen_topik = rng.sample(dokumen_topik, MAKS_DOKUMEN_BERTOPIC)

    print(f"Dokumen BERTopic (pra-oversampling): {len(dokumen_topik):,}  [FIX MR-2]")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if torch.cuda.is_available():
        print(f"VRAM sebelum encode: {torch.cuda.memory_allocated()/1e6:.1f} MB")

    embedder = SentenceTransformer(MODEL_EMBEDDING_BERTOPIC, device=dev)
    emb = embedder.encode(
        dokumen_topik, batch_size=UKURAN_BATCH_EMBED,
        show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True,
    )
    print(f"Shape embeddings: {emb.shape}")

    # [FIX LR-4] Hapus embedder segera setelah encoding
    del embedder; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"VRAM setelah hapus embedder: {torch.cuda.memory_allocated()/1e6:.1f} MB  [FIX LR-4]")

    umap_m = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine",
                   random_state=SEED_BERTOPIC, low_memory=True)
    hdb_m  = HDBSCAN(min_cluster_size=15, metric="euclidean",
                      cluster_selection_method="eom", prediction_data=True)
    vec_m  = CountVectorizer(ngram_range=(1,2), min_df=5)

    # [FIX MR-3] calculate_probabilities=False  [FIX Q3] nr_topics="auto"
    model_topik = BERTopic(
        language="multilingual",
        calculate_probabilities = BERTOPIC_HITUNG_PROB,  # False [FIX MR-3]
        nr_topics               = NR_TOPIK,              # "auto" [FIX Q3]
        verbose=True, embedding_model=None,
        umap_model=umap_m, hdbscan_model=hdb_m, vectorizer_model=vec_m,
    )
    id_topik, prob_topik = model_topik.fit_transform(dokumen_topik, emb)

    info = model_topik.get_topic_info()
    n_valid = len(info[info["Topic"] != -1])
    print(f"\nBERTopic selesai.")
    print(f"   Topik valid : {n_valid}")
    print(f"   Outlier (-1): {sum(1 for t in id_topik if t==-1):,}")
    print("\n10 topik teratas:")
    print(info.head(10).to_string(index=False))


In [ ]:
# ================================================================
# SEL 25 -- Visualisasi Distribusi Topik & Analisis per Label [A3]
# ================================================================
if AKTIFKAN_BERTOPIC and model_topik is not None:
    info_viz   = model_topik.get_topic_info()
    info_valid = info_viz[info_viz["Topic"] != -1].copy()

    # Plot 1: Top-N topik berdasarkan jumlah dokumen
    if len(info_valid) > 0:
        top_n  = min(15, len(info_valid))
        df_viz = info_valid.nlargest(top_n, "Count")
        plt.figure(figsize=(10, 5))
        plt.barh([str(n)[:45] for n in df_viz["Name"]], df_viz["Count"],
                  color=plt.cm.tab20(np.linspace(0,1,len(df_viz))))
        plt.xlabel("Jumlah Dokumen"); plt.ylabel("Topik")
        plt.title(f"Top {top_n} Topik BERTopic -- Seluruh Corpus",
                  fontsize=12, fontweight="bold")
        plt.gca().invert_yaxis(); plt.tight_layout()
        plt.savefig("distribusi_topik_bertopic.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Grafik distribusi topik disimpan: distribusi_topik_bertopic.png")

    # [A3] Analisis topik per label hoax/non-hoax
    df_analisis = train_df_pra_oversampling[["text","label"]].copy()
    n_cocok = min(len(df_analisis), len(id_topik))
    df_analisis = df_analisis.iloc[:n_cocok].copy()
    df_analisis["topic_id"] = id_topik[:n_cocok]
    df_analisis = df_analisis[df_analisis["topic_id"] != -1]

    if len(df_analisis) > 0 and len(info_valid) > 0:
        peta_nama = dict(zip(info_valid["Topic"], info_valid["Name"]))
        df_analisis["topic_name"] = df_analisis["topic_id"].map(peta_nama)
        top_k_a = min(10, len(info_valid))

        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        fig.suptitle("Distribusi Topik per Label -- [A3]", fontsize=13, fontweight="bold")

        for ax, lbl, judul, warna in [
            (axes[0], 0, "Topik Dominan -- Non-Hoax", "#16A34A"),
            (axes[1], 1, "Topik Dominan -- Hoax",     "#DC2626"),
        ]:
            subset = df_analisis[df_analisis["label"]==lbl]
            if len(subset) == 0:
                ax.text(0.5, 0.5, "Tidak ada data", ha="center"); continue
            hitung = subset["topic_name"].value_counts().head(top_k_a)
            ax.barh(hitung.index.str[:40], hitung.values, color=warna, alpha=0.85)
            ax.set_title(judul, fontsize=11, fontweight="bold")
            ax.set_xlabel("Jumlah Dokumen")
            ax.invert_yaxis(); ax.grid(axis="x", alpha=0.3)

        plt.tight_layout()
        plt.savefig("topik_per_label.png", dpi=150, bbox_inches="tight")
        plt.show()
        print("Analisis topik per label disimpan: topik_per_label.png  [A3]")
else:
    print("Model BERTopic belum tersedia, visualisasi dilewati.")


In [ ]:
# ================================================================
# SEL 26 -- Simpan Artefak BERTopic ke /content & Zip
# ================================================================
if AKTIFKAN_BERTOPIC and model_topik is not None:
    os.makedirs(DIR_OUTPUT_BERTOPIC, exist_ok=True)
    try:
        model_topik.save(DIR_OUTPUT_BERTOPIC, serialization="safetensors")
        print(f"BERTopic disimpan (safetensors) ke: {DIR_OUTPUT_BERTOPIC}")
    except TypeError:
        model_topik.save(DIR_OUTPUT_BERTOPIC)
        print(f"BERTopic disimpan (default) ke: {DIR_OUTPUT_BERTOPIC}")
    path_zip = shutil.make_archive(DIR_OUTPUT_BERTOPIC, "zip", DIR_OUTPUT_BERTOPIC)
    print(f"Zip: {path_zip}  ({os.path.getsize(path_zip)/1e6:.1f} MB)")
else:
    print("Model BERTopic belum tersedia; proses simpan dilewati.")


In [ ]:
# ================================================================
# SEL 27 -- Demo Inferensi Topik per Paragraf (BERTopic)
# ================================================================
# [FIX B1]   Regex DIPERBAIKI: r"(?:\r?\n){2,}"
#             Bug asli: double-escaped \r\n tidak pernah match
#             karena dalam raw string \r adalah dua karakter literal.
# [FIX LR-4] Re-init embedder karena sudah dihapus di SEL 24.
# ================================================================
import re as _re

# [FIX B1] Regex yang benar -- single backslash untuk \r dan \n
REGEX_PARAGRAF_DEMO = _re.compile("(?:\r?\n){2,}")


def pisahkan_paragraf_demo(teks: str) -> List[str]:
    mentah = str(teks).strip()
    if not mentah: return []
    bagian = [p.strip() for p in REGEX_PARAGRAF_DEMO.split(mentah) if p.strip()]
    return bagian if bagian else [mentah]


def label_dari_id_topik(obj_model, t_id: int) -> str:
    info  = obj_model.get_topic_info()
    baris = info[info["Topic"] == t_id]
    if len(baris) > 0 and "Name" in baris.columns:
        return str(baris.iloc[0]["Name"])
    kata = obj_model.get_topic(t_id)
    return " / ".join([w for w,_ in kata[:3]]) if kata else "topik_tidak_dikenal"


def skor_dari_prob(pb, t_id: int):
    if pb is None: return None
    if isinstance(pb, np.ndarray):
        if pb.ndim == 0: return float(pb)
        if 0 <= t_id < len(pb): return float(pb[t_id])
        return float(np.max(pb)) if pb.size > 0 else None
    return None


CONTOH_TOPIK = (
    "\nBeredar tautan bantuan tunai yang meminta OTP pengguna.\n\n"
    "Kementerian mengklarifikasi bahwa tautan tersebut palsu "
    "dan meminta warga cek kanal resmi.\n\n"
    "Pengguna lain melaporkan modus serupa berujung pada pencurian akun.\n"
)

if AKTIFKAN_BERTOPIC and model_topik is not None:
    daftar_par = pisahkan_paragraf_demo(CONTOH_TOPIK)
    print(f"Jumlah paragraf: {len(daftar_par)}  [FIX B1: regex diperbaiki]")

    if daftar_par:
        # [FIX LR-4] Re-init embedder karena sudah dihapus di SEL 24
        _emb_inf = SentenceTransformer(
            MODEL_EMBEDDING_BERTOPIC,
            device="cuda" if torch.cuda.is_available() else "cpu",
        )
        emb_par = _emb_inf.encode(
            daftar_par, batch_size=UKURAN_BATCH_EMBED,
            show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True,
        )
        del _emb_inf; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        id_par, prob_par = model_topik.transform(daftar_par, embeddings=emb_par)

        print("\nHasil topik per paragraf:")
        for idx, (teks_p, t_id) in enumerate(zip(daftar_par, id_par)):
            skor  = skor_dari_prob(prob_par[idx] if prob_par is not None else None, t_id)
            lbl   = label_dari_id_topik(model_topik, t_id)
            print(f"  Paragraf {idx}: Topik {t_id} | {lbl} | "
                  f"skor={f'{skor:.4f}' if skor is not None else 'N/A'}")
            print(f"  Teks: {teks_p}\n")
else:
    print("Demo BERTopic dilewati karena model belum di-fit.")


In [ ]:
# ================================================================
# SEL 28 -- Unggah Artefak BERTopic ke Hugging Face (OFF by default)
# ================================================================
if not AKTIFKAN_UNGGAH_BERTOPIC:
    print("AKTIFKAN_UNGGAH_BERTOPIC=False. Upload dilewati.")
elif not AKTIFKAN_BERTOPIC or model_topik is None:
    print("Model BERTopic belum tersedia.")
else:
    from huggingface_hub import login, HfApi
    from google.colab import userdata
    try:
        login(token=userdata.get("HF_TOKEN"))
    except Exception:
        print("Token HF_TOKEN tidak ditemukan.")
    api_bt = HfApi()
    api_bt.create_repo(repo_id=REPO_TOPIC_HF, private=False, exist_ok=True)
    print(f"Mengunggah BERTopic ke {REPO_TOPIC_HF} ...")
    api_bt.upload_folder(
        folder_path    = DIR_OUTPUT_BERTOPIC,
        repo_id        = REPO_TOPIC_HF,
        repo_type      = "model",
        commit_message = "Upload BERTopic V1-Fix (pra-oversampling, nr_topics=auto, regex-fix)",
    )
    print(f"Selesai: https://huggingface.co/{REPO_TOPIC_HF}")
